In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [16]:
data = pd.read_csv('Heart Prediction Quantum Dataset.csv')
print('Filas y columnas:', data.shape)

Filas y columnas: (500, 6)


In [17]:
antes = len(data)

data['BloodPressure'] = data['BloodPressure'].str.replace(',', '.', regex=False)
data['BloodPressure'] = data['BloodPressure'].astype(float)

In [18]:
data['Gender'] = data['Gender'].str.lower()
print('Tras lower:', data['Gender'].unique())

data['Gender'] = data['Gender'].str.rstrip(' {}')
print('Tras rstrip:', data['Gender'].unique())

data['Gender'] = data['Gender'].replace({'ombre': 'hombre'})
data['Gender'] = data['Gender'].map({'hombre': 1, 'mujer': 0})

Tras lower: <ArrowStringArray>
['hombre   ', 'mujer', 'ombre', 'mujer{', 'hombre}', 'hombre']
Length: 6, dtype: str
Tras rstrip: <ArrowStringArray>
['hombre', 'mujer', 'ombre']
Length: 3, dtype: str


In [19]:
data = data.drop_duplicates().dropna()
data['Gender'] = data['Gender'].astype(int)

print('Filas antes:', antes)
print('Filas despues:', len(data))
print('Gender final:', data['Gender'].unique())
data.dtypes

Filas antes: 500
Filas despues: 500
Gender final: [1 0]


Age                int64
Gender             int64
BloodPressure    float64
Cholesterol        int64
HeartRate          int64
HeartDisease       int64
dtype: object

In [20]:
X = data.drop(columns=['HeartDisease'])
y = data['HeartDisease']

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=50
)

In [36]:
from sklearn.neural_network import MLPClassifier

modelo = MLPClassifier(
    random_state=50,
    max_iter=500
)

In [23]:
capas = []

for c1 in range(5, 16):
    for c2 in range(10, 21):
        for c3 in range(5, 11):
            capas.append((c1, c2, c3))

In [24]:
len(capas) #11*11*6

726

In [37]:
from sklearn.model_selection import KFold

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=50
)

In [38]:
from sklearn.model_selection import GridSearchCV

parametros = {
    'hidden_layer_sizes': capas
}

grid = GridSearchCV(
    modelo,
    parametros,
    cv=kf,
    scoring='roc_auc'
)

In [39]:
grid.fit(X_train, y_train)

/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",MLPClassifier...ndom_state=50)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'hidden_layer_sizes': [(5, ...), (5, ...), ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the 

In [40]:
print("Mejores capas:", grid.best_params_)
print("AUC validación:", grid.best_score_)

Mejores capas: {'hidden_layer_sizes': (15, 13, 5)}
AUC validación: 0.8418735195205775


In [41]:
mejor_modelo = grid.best_estimator_

In [42]:
y_prob = mejor_modelo.predict_proba(X_test)[:, 1]
y_prob

array([0.73302603, 0.37122991, 0.24066493, 0.99480975, 0.70287755,
       0.29205074, 0.65875064, 0.84107966, 0.49635703, 0.95536753,
       0.28666002, 0.22481653, 0.99472112, 0.26636901, 0.91635764,
       0.87788667, 0.82529869, 0.1865831 , 0.91574459, 0.59417542,
       0.21196135, 0.72359709, 0.9436467 , 0.91800131, 0.99459147,
       0.98093906, 0.93483764, 0.39562303, 0.65482704, 0.96473031,
       0.25341827, 0.73184244, 0.26524526, 0.37044006, 0.23376188,
       0.62186552, 0.67992811, 0.51789428, 0.68887178, 0.94426714,
       0.28560631, 0.79728638, 0.99275105, 0.20511083, 0.40256437,
       0.3436317 , 0.95508035, 0.5687489 , 0.22620961, 0.65128271,
       0.8706541 , 0.44529804, 0.62932807, 0.97913899, 0.29350593,
       0.79195299, 0.29446123, 0.99865501, 0.99137227, 0.76225234,
       0.72395621, 0.95622248, 0.29607032, 0.51284959, 0.42854235,
       0.88096806, 0.90078255, 0.53470597, 0.9945112 , 0.96834715,
       0.19663234, 0.33157803, 0.84052029, 0.58913873, 0.85858

In [43]:
from sklearn.metrics import roc_auc_score

auc_test = roc_auc_score(y_test, y_prob)

print("AUC test:", auc_test)

AUC test: 0.7840886986967516
